# 01 种子入库

把 v2 种子（`data/raw/e09-paper-knowledge/seeds/`）写进 neo4j-e09，这是 e09 第一次写图。数据模型见 `docs/designs/v2/graph_model_v2.md`，种子格式见种子目录的 `README.md`。

写法沿用 e08（`04_neo4j`、`06_extract_prepare`）：读文件 → plan（只查不写）→ apply（单事务）→ 复核，重跑无副作用。和 e08 不同的是**身份怎么判断**：

- e08 按"主 Label + `name`"判断节点是否已存在；
- v2 按 NameKey 精确键（`intents_decompose.md` §3.2 的 D3）与 `identifiers`（`graph_model_v2.md` §2.4）解析。每条匹配规则都有四种结果：唯一命中、多重命中、无命中、数据冲突。plan 把冲突和查重候选显式列出来，第 5 节用不写库的小批次逐一演示。

plan 与 apply 对应 `Commit(delta, mode: dry_run | apply)` 的两种模式。这里只是种子的最小写入路径，不是入库表单的编译器：种子没有 Content、`FROM` 和 ResultUnit。

实现在 `src/e09/`（`seed`、`namekey`、`schema`、`graph`、`ids`），本 notebook 只导入、展示并演示失败路径。可复现的写入走 `make seed`（`python -m e09.seed`，按 `SEED_ORDER` 写入，有冲突或复核失败时返回非零）；两者调用同一套代码，结果相同。

## 1. 预设参数

路径与连接参数来自 `e09.config`，与 `make check` 共用。

In [ ]:
import pandas as pd
import yaml

from e09.config import NEO4J_DB, NEO4J_URI, SEEDS

RESET = False   # True：清空整个库（DETACH DELETE 全部节点）后从空图重演；仅限 neo4j-e09 只放种子时使用
print(NEO4J_URI, SEEDS)

## 2. 读种子，做与库无关的检查

`check_seed` 只看文件本身，不查库：

- `labels` 恰好两个：主 Label（Entity / Concept）与 kind，kind 在种子允许的范围内（种子 `README.md`）；
- 不自带 `id`（入库时分配）；必填 `name`，Concept 必填 `definition`，Entity 必填 `description`；Metric 的 `direction` 为 `lower` / `higher`；
- `identifiers` 每项是已声明命名空间的 `"<namespace>:<value>"`；唯一命名空间（`arxiv`、`doi`、`s2`）批内不能重复；
- `name` 与 `aliases` 各算一个精确键 `规范化名称|kind|scope`，批内同一个键不能注册给两个节点；
- 关系的端点在本文件内，类型与端点的主 Label、kind 符合 `graph_model_v2.md` §6。

精确键的规范化是 `name-key-v1`（`intents_decompose.md` §3.2）：NFC；由 Unicode White_Space 字符组成的连续空白折叠为一个空格，再去掉首尾空白；保留大小写与标点。所以 `BM-25` 与 `BM25`、`BERT` 与 `bert` 是不同的键。种子都是全局 scope；切分（Split）的 scope 是父数据集，种子里没有。

`aliases` 是表单字段，入库时注册为 NameKey，不写进对象；`sources`、`verified` 只留在种子文件里。

实现在 `e09.seed`（检查）、`e09.namekey`（name-key-v1）与 `e09.schema`（kind、命名空间、关系端点）；这里只导入。

In [ ]:
from e09.namekey import NORMALIZER
from e09.seed import SEED_ORDER, check_seed, load_seed

In [ ]:
seeds = {f: load_seed(SEEDS / f) for f in SEED_ORDER}   # 固定顺序：id 按写入顺序分配
print(NORMALIZER)
pd.DataFrame([{"file": name, "nodes": len(b["nodes"]), "name_keys": sum(len(n["keys"]) for n in b["nodes"].values()),
               "relationships": len(b["rels"])} for name, b in seeds.items()]).set_index("file")

## 3. 连接 neo4j-e09，建约束

neo4j-e08 与 neo4j-e09 端口相同，写之前先用 `env_check.check_graph()` 确认没有连到 v1 库。

约束全部 `IF NOT EXISTS`，重跑无副作用：

- `Entity.id`、`Concept.id` 唯一；
- `NameKey.key` 唯一。D3 的"写入时唯一"由这条约束兜底，这也是 NameKey 不存成对象上 `aliases` 列表的原因：唯一约束作用于整个属性值，不作用于列表元素（`graph_model_v2.md` 第 5 节）。

唯一命名空间的 `identifiers` 是对象上的字符串列表，数据库约束管不到，由 plan 检查、apply 在同一事务内复查。

In [ ]:
from e09.env_check import check_graph
from e09.graph import driver, ensure_schema, q   # driver 关掉 UNRECOGNIZED 类提示：空库时查不存在的 Label 不报警

assert not check_graph(), "不是 v2 库，停止"
driver.verify_connectivity()
ensure_schema()   # e09.schema.CONSTRAINTS，全部 IF NOT EXISTS


def library_status():
    total = q("MATCH (n) RETURN count(n) AS c")[0]["c"]   # 每个节点带两个 Label，不能按 Label 计数相加
    kinds = q("MATCH (n) RETURN labels(n)[0] AS a, labels(n)[1] AS b, count(*) AS n")
    rels = q("MATCH ()-[r]->() RETURN type(r) AS type, count(*) AS n ORDER BY type")
    print(f"节点 {total}，关系 {sum(r['n'] for r in rels)}")
    if kinds:   # Neo4j 不保证 labels(n) 的顺序，这里按主 Label 摆正
        rows = [{"label": ":".join(sorted([r["a"], r["b"]] if r["b"] else [r["a"]],
                                          key=lambda l: l not in ("Entity", "Concept"))), "n": r["n"]} for r in kinds]
        display(pd.DataFrame(rows).groupby("label").sum().T)
    if rels:
        display(pd.DataFrame(rels).set_index("type").T)


if RESET:
    deleted = driver.execute_query("MATCH (n) DETACH DELETE n", database_=NEO4J_DB).summary.counters.nodes_deleted
    print(f"RESET：删除 {deleted} 个节点")

display(pd.DataFrame(q("SHOW CONSTRAINTS YIELD name, labelsOrTypes, properties")))
library_status()

## 4. 写入：plan → apply → 复核

**`plan_seed`**（对应 `Commit` 的 `dry_run`）只查不写，对每个节点做写入模式的解析：

1. **对象身份看 `name` 的精确键。** 命中某个对象就是该对象（D3 唯一键直接 resolved），属性有差异的列为"更新"；未命中就是新对象。
2. **每个精确键（`name` 与各 alias）分四种：**
   - 未注册 → 新键；
   - 已注册给同一对象 → 不变（同键同对象的重复注册是幂等的）；
   - 已注册给别的对象 → **冲突**；
   - 已隔离为 `ambiguous` → **冲突**，暂停精确解析。

   名称未命中、某个 alias 却指向已有对象 X，也属于冲突：它是 X 的新名称，还是另一个对象，要交给外部判断。
3. **`identifiers` 按命名空间处理（§2.4）：**
   - 唯一命名空间被别的对象占用 → **冲突**。
   - 非唯一的 `url` 被别的对象（库里的，或本批里的）同时持有 → **查重候选**，不阻止写入。这和 Resolve 写入模式的约定一致：多重命中只供外部判断"新对象是其中之一，还是同一仓库里的另一个对象"，不直接复用。种子里同一仓库下的多个数据集是作者有意分开建的，所以查重候选在这里只列出、不处理。
4. 关系以 `(from, type, to)` 为身份；两端都已在库里时查一次是否已存在，语义对称的不分方向。

**`apply_plan`**（对应 `apply`）：有冲突就整批拒绝；没有变化就跳过。否则在**一个写事务**里完成以下步骤，中途出错整批回滚：

- 建新对象；
- 已有对象 `SET n += props`；
- 注册 NameKey；
- 复查唯一标识；
- 建关系。

NameKey 先 `MERGE`，再在同一事务里确认这个键只指向本对象、且仍是 `active`，确保唯一检查与写入原子执行。写完再 plan 一次，必须没有任何待写内容。

更新语义和 e08 一样是 `SET n += props`：从种子里删掉一个属性或 alias **不会**删掉库里的值，NameKey 的撤销与改指向（对象合并，Q4）不在这里。

**id** 形如 `<kind 前缀>_<4 位序号>`（如 `task_0001`、`dataset_0001`），由 `IdAllocator` 从图中已用的最大序号起顺序分配；分配后写入失败只会留下空号，不会重号。

实现在 `e09.seed`（plan、apply）与 `e09.ids`（id 分配）；这里只导入，另加一个展示 plan 的视图：`summary` 的文字摘要之外，把查重候选与冲突摆成表。

In [ ]:
from e09.ids import IdAllocator
from e09.seed import Plan, apply_plan, plan_seed, summary


def show_plan(p: Plan):
    print(summary(p))
    if p.candidates:
        display(pd.DataFrame(p.candidates).groupby("identifier").agg(
            new=("ref", list), in_graph=("in_graph", lambda s: sorted({x for v in s for x in v}))))
    if p.conflicts:
        display(pd.DataFrame(p.conflicts))

### 4.1 时序预测种子 `tsf.yml`

第一次运行时应全部是"新建"。ETT 与 ETTh1–ETTm2 共用 `url:https://github.com/zhouhaoyi/ETDataset`，会作为查重候选列出；它们是种子作者有意分开建的数据集，不影响写入。

In [ ]:
ids = IdAllocator()
print("分配起点", {k: v for k, v in ids.last.items() if v} or "（空库）")

p_tsf = plan_seed(seeds["tsf.yml"])
show_plan(p_tsf)

In [ ]:
ref2id = {"tsf.yml": apply_plan(p_tsf, ids)}
library_status()

### 4.2 RAG 种子 `rag.yml`

两个文件没有共享节点，也没有跨文件关系。查重候选来自同一 url 下的多个数据集与模型（如 UltraDomain、OpenAI 的 API 模型）。

In [ ]:
p_rag = plan_seed(seeds["rag.yml"])
show_plan(p_rag)

In [ ]:
ref2id["rag.yml"] = apply_plan(p_rag, ids)
library_status()

### 4.3 重跑

同一批再 plan 一次：全部不变、没有查重候选（只有新对象需要查重），解析到的 id 与写入时一致。

In [ ]:
for name, b in seeds.items():
    again = plan_seed(b)
    assert again.empty and not again.conflicts and not again.candidates, name
    assert again.target == ref2id[name], f"{name}：重跑解析到的 id 与写入时不同"
    print(f"{name}：{len(again.same_nodes)} 个对象、{again.same_keys} 个精确键、{len(again.same_rels)} 条关系均已存在")

## 5. 失败路径：只 plan，不写库

下面几个小批次写在 notebook 里，不属于种子。每个批次只跑 plan；会被拒绝的批次再调用一次 `apply_plan`，以确认它确实不写库。

**(a) 别名已注册给另一个对象。** 新方法的 alias `DMS` 已经是 `Direct multi-step forecasting` 的精确键。plan 报 `key-taken`，apply 整批拒绝，库内节点数不变。外部可以选择换一个 alias、改用经确认的作用域，或者判定这其实就是已有对象。

In [ ]:
bad_alias = check_seed({"nodes": [{
    "ref": "m_demo", "labels": ["Concept", "Method"],
    "properties": {"name": "Demo linear forecaster", "aliases": ["DMS"],
                   "definition": "Demonstration only: a node whose alias collides with an existing method."}}]},
    "bad_alias（inline）")
before = q("MATCH (n) RETURN count(n) AS c")[0]["c"]
p_bad = plan_seed(bad_alias)
show_plan(p_bad)
assert apply_plan(p_bad, ids) == {} and q("MATCH (n) RETURN count(n) AS c")[0]["c"] == before

**(b) 唯一命名空间在批内重复。** 两个数据集带同一个 `arxiv` 标识，`check_seed` 读文件时就拒绝，不必查库。如果其中一个已经在库里，plan 会报 `identifier-taken`；种子里没有唯一命名空间的标识，这条路径要等论文入库后才会真正走到。

In [ ]:
try:
    check_seed({"nodes": [
        {"ref": f"d_demo{j}", "labels": ["Entity", "Dataset"],
         "properties": {"name": f"Demo dataset {j}", "identifiers": ["arxiv:0000.00000"],
                        "description": "Demonstration only."}} for j in (1, 2)]}, "dup_arxiv（inline）")
except AssertionError as e:
    print("拒绝：", e)

**(c) 非唯一标识的多重命中。** 一个新数据集也挂在 `zhouhaoyi/ETDataset` 下：库里已有 5 个对象持有这个 url。plan 把它们全部列为查重候选，没有冲突，所以 apply 会照常写入（这里不写）。它是这 5 个之一，还是同一仓库里的第 6 个数据集，要由外部判断；读取模式下的 Resolve 则会返回 `stage=id, status=ambiguous`，再与名称取交集（§2.4）。

In [ ]:
p_url = plan_seed(check_seed({"nodes": [{
    "ref": "d_demo", "labels": ["Entity", "Dataset"],
    "properties": {"name": "Demo ETT subset", "identifiers": ["url:https://github.com/zhouhaoyi/ETDataset"],
                   "description": "Demonstration only: a new dataset published in an existing repository."}}]},
    "same_url（inline）"))
show_plan(p_url)
assert not p_url.conflicts and len(p_url.candidates) == 1

**(d) 名称命中即同一对象。** 批次里写一个 `MSE`，换了 definition：`MSE|Metric|global` 已注册，plan 把它解析为库里的 MSE，列为"更新"并给出字段差异，而不是新建一个同名 Metric。要不要接受这个更新，由提交者看完差异后决定。

In [ ]:
p_mse = plan_seed(check_seed({"nodes": [{
    "ref": "met_mse", "labels": ["Concept", "Metric"],
    "properties": {"name": "MSE", "direction": "lower", "definition": "Demonstration only: a revised definition."}}]},
    "mse_update（inline）"))
show_plan(p_mse)
assert list(p_mse.upd_nodes) == ["met_mse"] and not p_mse.new_nodes

## 6. 库内结果

- 对象上不存 `aliases`；`Get` 读取时由 NameKey 装配：`raw` 与对象 `name` 不同的键就是 alias。
- 每个对象都有一条与自己 `name` 对应的 NameKey；NameKey 总数等于种子里规范化去重后的称呼数。
- 共用同一个 url 的对象组：这是之后检验 Resolve 多重命中的现成用例。

In [ ]:
expect_keys = sum(len(n["keys"]) for b in seeds.values() for n in b["nodes"].values())
checks = q("""
    CALL () { MATCH (n) WHERE n.aliases IS NOT NULL RETURN count(n) AS with_aliases_list }
    CALL () { MATCH (n:Entity|Concept) WHERE NOT EXISTS { MATCH (k:NameKey)-[:NAMES]->(n) WHERE k.raw = n.name }
              RETURN count(n) AS without_name_key }
    CALL () { MATCH (k:NameKey) RETURN count(k) AS name_keys }
    RETURN with_aliases_list, without_name_key, name_keys""")[0]
print(checks, "期望 NameKey", expect_keys)
assert checks == {"with_aliases_list": 0, "without_name_key": 0, "name_keys": expect_keys}

In [ ]:
# Get 视角：对象 + 由 NameKey 装配的 aliases
pd.DataFrame(q("""
    MATCH (n:Entity|Concept)
    OPTIONAL MATCH (k:NameKey)-[:NAMES]->(n) WHERE k.raw <> n.name
    WITH n, collect(k.raw) AS aliases
    RETURN n.id AS id, [l IN labels(n) WHERE NOT l IN ['Entity', 'Concept']][0] AS kind, n.name AS name, aliases
    ORDER BY id LIMIT 12"""))

In [ ]:
# 共用同一 url 的对象组
pd.DataFrame(q("""
    MATCH (n:Entity) UNWIND n.identifiers AS i
    WITH i, collect(n.id + ' ' + n.name) AS objects WHERE i STARTS WITH 'url:' AND size(objects) > 1
    RETURN i AS identifier, size(objects) AS n, objects ORDER BY i"""))

这里还没覆盖的：

- **Resolve（读取模式）**：id 级多重命中后与名称取交集，交集唯一则 resolved，否则保留 ambiguous（§2.4）。上面的 url 组可以直接作为用例。
- **`ambiguous` 隔离**：只在批量导入或事后核查发现既有冲突时出现，需要一个核查流程，种子入库走不到。
- **NameKey 的撤销与改指向**：删 alias、合并对象（Q4）。
- **规范化的 Unicode 版本**：设计写的是 15.1，Python 3.12 的 `unicodedata` 是 15.0，版本号已如实记进 `normalizer_ref`。对现有种子的 ASCII 名称没有影响。
- **Content 一侧**：`Material`、`FROM`、Experiment 与 ResultUnit 等论文入库时再做，依赖入库表单的剩余三项（实例范围、方法变体、条件槽）。

In [ ]:
library_status()
driver.close()